# Reproducing Chen, He et al. (2026) from dandiset 001933

Paper: Chen C, He Q, Tombesi G, et al. (2026). Leucine-rich repeat kinase 2 impairs the release sites of
Parkinson's disease vulnerable dopamine axons. Nature Communications.
[doi:10.1038/s41467-026-75194-3](https://doi.org/10.1038/s41467-026-75194-3). This attempt used the open access
full text (CC BY) and its Supplementary Information.

Dandiset: [DANDI:001933](https://dandiarchive.org/dandiset/001933/0.260825.1457), version 0.260825.1457. The paper
does not name the dandiset or a version (its data statement points to Zenodo, PRIDE and the BioImage Archive). This is
the only published version, and it holds the same 27 files as the draft (checked in the data check), so it is pinned.

What the dandiset holds: one session per mouse of GRAB-DA3m fiber photometry in the dorsal striatum during optogenetic
stimulation of SNc dopamine neurons, 15 Anxa1-iCre mice (dorsolateral striatum) and 12 Calb1-Cre mice (dorsomedial
striatum), LRRK2-WT and LRRK2-G2019S (GS). This is the data of Figure 6 and Supplementary Figures 7 and 8. The
immunostaining, proteomics, western blots and super-resolution imaging of Figures 1 to 5 are not in it.

Outside input: the paper's source data file (Zenodo [10.5281/zenodo.19930567](https://doi.org/10.5281/zenodo.19930567),
`Source data.xlsx`, md5 checked below), used only in the claim `source-data-link` to test whether the dandiset's
recordings are those behind the published values. Every other claim runs on the dandiset alone. The paper's analysis
code was read at [DombeckLab/lrrk2_photometry_analysis](https://github.com/DombeckLab/lrrk2_photometry_analysis),
commit d34558f (the Zenodo release 10.5281/zenodo.20244434 is commit 1b551d1, which differs only by a licence file).

| Claim | Paper | Tested |
|---|---|---|
| `source-data-link` | Source data of Fig. 6, Supp. Figs 7 and 8 can be recomputed from the dandiset | tested (with the source data) |
| `fig6e-anxa1-evoked-reduced` | Anxa1 evoked release lower in GS at 0.5 mW (p = 1.10e-20) | tested |
| `fig6e-anxa1-evoked-per-mouse` | 7 of 8 Anxa1 GS mice reduced; 3743 not | tested |
| `supp7c-anxa1-evoked-all-powers` | Reduction at 0.1, 0.5, 1 and 4 mW | tested |
| `fig6f-anxa1-spontaneous-reduced` | Anxa1 spontaneous transients smaller in GS (1064 against 798) | tested |
| `fig6f-anxa1-spontaneous-per-mouse` | 6 of 8 Anxa1 GS mice reduced; 4651 and 3743 not | tested |
| `supp8c-calb1-spontaneous-not-reduced` | Calb1 spontaneous transients not reduced (p = 0.98) | tested |
| `supp8e-calb1-evoked-reduced` | Calb1 evoked release reduced at four powers | tested |
| `supp8e-calb1-few-mice-reduced` | Only 2 of 7 Calb1 GS mice reduced | tested |
| `cohens-d-anxa1-larger` | Cohen's d at 0.5 mW, Anxa1 2.05 against Calb1 0.76 | tested |
| `fig1d-lrrk2-enriched-vulnerable` and seven other claims of Figs 1 to 5 | Imaging, proteomics, western blots | blocked: not in the dandiset |

Method choices are fixed in the claims file, `claims/001933/10.1038-s41467-026-75194-3.yaml`. In short: the signal is
the dandiset's processed dF/F, the trace the paper's code analyses; the evoked integral follows the Methods (the
light-on period), with the code's baseline; spontaneous transients follow the Methods and `transient_by_threshold.m`.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import lrrk2

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
sessions = lrrk2.load_all()
by_mouse = {s["mouse"]: s for s in sessions}
print(f"loaded {len(sessions)} files from dandiset {lrrk2.DANDISET_ID} version {lrrk2.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")
# No step of this analysis draws random numbers, so no generator is needed.

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
info = pd.DataFrame([{
    "mouse": s["mouse"], "subject_id": s["subject_id"], "line": s["line"], "group": s["group"],
    "genotype": s["genotype"], "sex": s["sex"], "date_of_birth": s["date_of_birth"][:10],
    "session_start": s["session_start"][:19],
    "age_months": round((pd.Timestamp(s["session_start"][:10]) - pd.Timestamp(s["date_of_birth"][:10])).days / 30.44, 1),
    "recorded_in": s["location"][s["dff_region"]], "species": s["species"], "strain": s["strain"],
    "dff_md5": s["md5"][:12]} for s in sessions]).sort_values(["line", "group", "mouse"])
print("Files (one session each) by Cre line and genotype:")
print(pd.crosstab(info.line, info.group, margins=True).to_string())
print("\nPaper: Anxa1 8 G2019S and 7 WT mice (Figure 6C); Calb1 7 G2019S and 5 WT mice (Supplementary Figure 8D),"
      "\nafter excluding one Anxa1 G2019S, one Calb1 G2019S and two Calb1 WT mice (Methods).")
paper_anxa1_gs = [4885, 4651, 4652, 3742, 3743, 4004, 4005, 4007]  # mouse IDs in the Figure 6E legend
ours = sorted(info[(info.line == "Anxa1") & (info.group == "GS")].mouse)
print("Anxa1 G2019S mouse IDs in the Figure 6E legend found in the dandiset:", sorted(paper_anxa1_gs) == ours, ours)
print("\nSex by line and genotype:")
print(pd.crosstab([info.line, info.group], info.sex).to_string())
print("\nAge at the session (months), by line: ", info.groupby("line").age_months.agg(["min", "max"]).to_dict("index"))
print("Recording site by line:", info.groupby("line").recorded_in.unique().to_dict())
print("Species and strain:", info.species.unique().tolist(), info.strain.unique().tolist())
print("\nDistinct dF/F traces (md5 of the full series):", info.dff_md5.nunique(), "of", len(info),
      "; distinct session start times:", info.session_start.nunique())
print("\n", info.to_string(index=False))

# The pinned version against the draft: same files and content?
from dandi.dandiapi import DandiAPIClient
client = DandiAPIClient()
def digests(version):
    return {a.path: (a.size, a.get_raw_metadata()["digest"].get("dandi:dandi-etag")) for a in client.get_dandiset(lrrk2.DANDISET_ID, version).get_assets()}
pinned, draft = digests(lrrk2.DANDISET_VERSION), digests("draft")
print(f"\nAssets: pinned version {len(pinned)}, draft {len(draft)}; identical paths, sizes and etags: {pinned == draft}")

The files hold one session each of 15 Anxa1 mice (8 GS, 7 WT) and 12 Calb1 mice (7 GS, 5 WT), the paper's counts after
its exclusions, and the Anxa1 GS mouse IDs are those of the Figure 6E legend. The four excluded mice are not in the
dandiset, which does not say that they were left out. Ages at the session are 6.3 to 7.6 months, consistent with
surgery at 6 months and recording four weeks later. Anxa1 mice are recorded in the dorsolateral and Calb1 mice in the
dorsomedial striatum, as the paper states. No two files share a dF/F trace or a start time, so no recording is
deposited twice. The pinned version and the draft hold the same files.

In [ ]:
rows = []
for s in sessions:
    t, x = s["t"], s["dff"]
    dt = np.diff(t) * 1000
    rows.append({"mouse": s["mouse"], "line": s["line"], "group": s["group"], "samples": len(t),
                 "first_s": round(t[0], 2), "minutes": round((t[-1] - t[0]) / 60, 1),
                 "dt_med_ms": round(np.median(dt), 2), "dt_min_ms": round(dt.min(), 2), "dt_max_ms": round(dt.max(), 2),
                 "dff_min": round(x.min(), 1), "dff_p8": round(np.percentile(x, 8), 2), "dff_max": round(x.max(), 1),
                 "dff405_min": round(s["dff405"].min(), 1), "dff405_max": round(s["dff405"].max(), 1),
                 "max_diff_recomputed": round(np.abs(x - s["dff_recomputed"]).max(), 4),
                 "raw_lag_samples": s["raw_lag"], "raw_r_at_lag0": round(s["raw_r"], 3),
                 "raw_470_samples_per_bin": round(s["raw_n"] / len(t), 2)})
series = pd.DataFrame(rows).sort_values(["line", "group", "mouse"])
print("dF/F series: stated unit", repr(sessions[0]["dff_unit"]), "; description:", sessions[0]["dff_description"].strip())
print(series.to_string(index=False))
print("\nSampling interval over all files (ms): median", series.dt_med_ms.median(), "min", series.dt_min_ms.min(),
      "max", series.dt_max_ms.max())
print("Recording length (min):", series.minutes.min(), "to", series.minutes.max(), "(Methods: 20 min)")
print("Largest difference between the stored dF/F and the code's df_f plus 20-sample moving mean recomputed from the",
      "stored corrected signal (% dF/F):", series.max_diff_recomputed.max())
print("Lag that best aligns the raw 470 nm signal, averaged into each 100 Hz bin, with the corrected signal:",
      sorted(series.raw_lag_samples.unique()), "; correlation at lag 0 from", series.raw_r_at_lag0.min(), "to",
      series.raw_r_at_lag0.max())

The processed series are regular at 10 ms (the shortest interval, 9.5 ms, comes from the 2 kHz grid) and last 20.4 to
25.4 min, a little longer than the Methods' 20 min. The dF/F series is stated in `a.u.`, but its description and its
values are % dF/F: the code's `df_f` and 20-sample moving mean applied to the stored corrected signal give the stored
series to within 0.026 % dF/F. The raw 470 nm signal, averaged into the 100 Hz bins, aligns with the corrected signal
at lag 0 in every file (correlation 0.77 to 0.99), so raw and processed series share a clock. About 8 of the 20 raw
samples per bin are kept for 470 nm, as the demultiplexing describes.

In [ ]:
code_letter = {0.1: "a", 0.25: "b", 0.5: "c", 1.0: "d", 1.5: "e", 2.0: "f", 3.0: "g", 4.0: "h"}
rows = []
for s in sessions:
    p = s["power_in_mW"]
    dur = s["stop_time"] - s["start_time"]
    paper = np.isin(p, lrrk2.PAPER_POWERS)
    gaps = np.diff(s["start_time"])
    rows.append({"mouse": s["mouse"], "line": s["line"], "group": s["group"], "epochs": len(p),
                 "per_power": "/".join(str((p == q).sum()) for q in sorted(set(p))),
                 "powers": len(set(p)), "first_8_codes": "".join(code_letter[q] for q in p[:8]),
                 "paper_powers_in_half": 1 if paper[:32].all() else (2 if paper[32:].all() else 0),
                 "train_s_paper": round(np.median(dur[paper]), 3), "pulse_ms_paper": s["pulse_length_in_ms"][paper][0],
                 "period_ms_paper": s["period_in_ms"][paper][0],
                 "train_s_other": round(np.median(dur[~paper]), 3), "pulse_ms_other": s["pulse_length_in_ms"][~paper][0],
                 "period_ms_other": s["period_in_ms"][~paper][0],
                 "min_gap_s": round(gaps.min(), 1), "median_gap_s": round(np.median(gaps), 1),
                 "wavelength": s["wavelength_in_nm"][0], "first_onset_s": round(s["start_time"][0], 1),
                 "onset_minus_series_start_s": round(s["start_time"][0] - s["t"][0], 1)})
opto = pd.DataFrame(rows).sort_values(["line", "group", "mouse"])
print("Epochs table description (first file):", sessions[0]["opto_description"])
print("\nPowers in the files (mW):", sorted(set(np.concatenate([s["power_in_mW"] for s in sessions]))))
print(opto.to_string(index=False))
print("\nTrain types carrying the paper's four powers, by line and genotype:")
print(opto.groupby(["line", "group", "train_s_paper", "pulse_ms_paper", "period_ms_paper"]).size().to_string())
odd = [(s["mouse"], j) for s in sessions for j in np.where((s["pulse_length_in_ms"] == 0) | np.isnan(s["period_in_ms"]))[0]]
print("\nEpochs with a pulse length of 0 or no period (mouse, epoch, power, length in s, pulses):",
      [(m, int(j), by_mouse[m]["power_in_mW"][j], round(by_mouse[m]["stop_time"][j] - by_mouse[m]["start_time"][j], 2),
        int(by_mouse[m]["number_pulses_per_pulse_train"][j])) for m, j in odd])
print("Shortest gap between onsets in any file (s):", opto.min_gap_s.min(), "(Methods: at least 20 s)")
gap_all = np.concatenate([np.diff(s["start_time"]) for s in sessions])
print("Gaps between onsets over all files (s): 5th, 50th, 95th percentiles", np.percentile(gap_all, [5, 50, 95]).round(1),
      "; fraction under 20 s:", round(float((gap_all < 20).mean()), 3))

Every file holds 64 trains, 8 at each of 8 powers from 0.1 to 4 mW; the paper uses four of them (0.1, 0.5, 1 and 4 mW).
The table's own description gives 8 ms on and 8 ms off for 500 ms and only those four powers, but the table holds two
kinds of train. In Anxa1 files one half of the session has 9 ms pulses every 10 ms for 0.33 s, the other 8 ms pulses
every 16 ms for 0.51 s; Calb1 files hold only the second kind. The paper's four powers fall on 0.51 s trains in 25
files and on 0.33 s trains in two Anxa1 GS mice (3742 and 4885), so these two mice were stimulated differently from
every WT mouse; a variant of `fig6e-anxa1-evoked-reduced` leaves them out. One train each in mice 4652 and 740 has a
pulse length of 0, no period and one pulse, though it lasts 0.51 s like its neighbours, so its pulse columns are
wrong. Trains of the same power come in pairs 15.3 to 15.5 s apart, and 51% of the gaps are under 20 s, where the
Methods and the table description say at least 20 s.

In [ ]:
rows = []
for s in sessions:
    shifted = dict(s, onset=s["onset"] + 1000, offset=s["offset"] + 1000)  # 10 s later, between trains
    true = lrrk2.evoked_integrals(s, "fixed")
    moved = lrrk2.evoked_integrals(shifted, "fixed")
    iso = lrrk2.evoked_integrals(s, "fixed", trace="dff405")
    traces = np.array([s["dff"][i - 100:i + 300] for i in s["onset"] if i + 300 <= len(s["dff"])])
    mean_trace = traces.mean(0)
    ok = ~np.isnan(true)
    rows.append({"mouse": s["mouse"], "line": s["line"], "group": s["group"],
                 "median_at_onsets": np.nanmedian(true), "median_10s_later": np.nanmedian(moved),
                 "median_405_at_onsets": np.nanmedian(iso),
                 "peak_latency_ms": (np.argmax(mean_trace) - 100) * 10,
                 "rise_before_onset": mean_trace[90:100].mean() - mean_trace[:90].mean(),
                 "spearman_power": stats.spearmanr(s["power_in_mW"][ok], true[ok])[0],
                 **{f"{q}mW": np.nanmedian(true[s["power_in_mW"] == q]) for q in sorted(set(s["power_in_mW"]))}})
clock = pd.DataFrame(rows).sort_values(["line", "group", "mouse"])
print("Integral from 0 to 0.5 s after onset minus the mean of -1.0 to -0.1 s (% dF/F x s), median over each file's 64",
      "trains, at the table's onsets, 10 s later, and for the 405 nm control; latency of the peak of the mean trace;",
      "Spearman correlation of the integral with the labelled power; median integral per labelled power.")
print(clock.round(2).to_string(index=False))
print("\nFiles where the median at onsets exceeds the median 10 s later:", int((clock.median_at_onsets > clock.median_10s_later).sum()),
      "of", len(clock), "; Wilcoxon p =", f"{stats.wilcoxon(clock.median_at_onsets, clock.median_10s_later).pvalue:.2g}")
print("Peak latency (ms) from", clock.peak_latency_ms.min(), "to", clock.peak_latency_ms.max(),
      "; mean rise in the 0.1 s before onset from", clock.rise_before_onset.min().round(2), "to", clock.rise_before_onset.max().round(2))
print("Spearman correlation with power, median by line:", clock.groupby("line").spearman_power.median().round(2).to_dict())

At the table's onsets the 0 to 0.5 s integral has a median of 13 to 260 % dF/F x s per file; 10 s later, between
trains, it is near zero in every file (lower in 27 of 27, Wilcoxon p = 1.5e-08), and the 405 nm control shows no
response (medians -0.74 to 1.15). The mean response peaks 160 to 370 ms after onset. The mean trace rises by up to
25 % dF/F in the 0.1 s before onset: the 20-sample moving mean of the stored dF/F spreads the response about 0.1 s
earlier, which is why the code's baseline ends 0.1 s before onset. The epochs table and the photometry share a clock.

The response grows with the labelled power in every Calb1 file (Spearman 0.76 to 0.94) but not in many Anxa1 files
(median 0.06). In Anxa1 files where 1.5, 2 and 3 mW are given as 0.33 s trains, those responses fall below the 0.5
and 1 mW ones given as 0.51 s trains (for example mouse 4401). The power labels are consistent with the data in Calb1
files; in Anxa1 files the data alone cannot confirm them. The labels of the four powers the paper uses are confirmed
against its source data under `source-data-link`.

## Claims

### source-data-link

> The dandiset's recordings are those behind the paper's Figure 6 and Supplementary Figures 7 and 8, and the paper's
> published per-stimulation traces, evoked integrals and spontaneous transient integrals can be recomputed from them.

Records are matched by mouse ID, group, labelled power and the rank of the stimulation within its power. The source
data file is read in memory from Zenodo.

In [ ]:
md5, src = lrrk2.load_source_data()
print("Source data file md5:", md5, "; matches the Zenodo record:", md5 == lrrk2.SOURCE_DATA_MD5)
print("Records per table:", {k: len(v) for k, v in src.items()})
line_of = {"6C": "Anxa1", "6E": "Anxa1", "6F": "Anxa1", "S7B": "Anxa1", "S7C": "Anxa1",
           "S8C": "Calb1", "S8D": "Calb1", "S8E": "Calb1"}

# (1) Mice
mice = {(k, d["mouse"], d["group"]) for k, v in src.items() for d in v}
part1 = pd.DataFrame([{"table": k, "mouse": m, "group": g, "in_dandiset": m in by_mouse,
                       "same_group": m in by_mouse and by_mouse[m]["group"] == g,
                       "same_line": m in by_mouse and by_mouse[m]["line"] == line_of[k]} for k, m, g in mice])
part1["match"] = part1.in_dandiset & part1.same_group & part1.same_line
mouse_level = part1.groupby("mouse").match.all()
print(f"\n(1) Mice: {int(mouse_level.sum())} of {len(mouse_level)} source-data mice are in the dandiset with the same genotype and line;",
      "dandiset mice not in the source data:", sorted(set(by_mouse) - set(mouse_level.index)))


def rank_match(records):
    """Pair each source record with the dandiset epoch of the same mouse and power and the same rank."""
    seen, out = {}, []
    for d in records:
        key = (d["mouse"], d["power"])
        r = seen.get(key, 0)
        seen[key] = r + 1
        s = by_mouse[d["mouse"]]
        idx = np.where(np.isclose(s["power_in_mW"], d["power"]))[0]
        out.append((d, s, idx[r], idx))
    return out


# (2) Triggered traces
rows = []
for key in ("6C", "S7B", "S8D"):
    for d, s, j, idx in rank_match(src[key]):
        ours = lrrk2.code_chunk(s, j)
        rows.append({"table": key, "line": s["line"], "mouse": d["mouse"], "power": d["power"],
                     "max_diff": np.nanmax(np.abs(ours - d["trace"])), "nan_here": int(np.isnan(ours).sum()),
                     "r": np.corrcoef(ours[~np.isnan(ours)], d["trace"][~np.isnan(ours)])[0, 1],
                     "slope": np.polyfit(ours[~np.isnan(ours)], d["trace"][~np.isnan(ours)], 1)[0]})
part2 = pd.DataFrame(rows)
part2["match"] = part2.max_diff < 1e-6
print("\n(2) Triggered traces equal to the dandiset's dF/F from 1 s before the onset of the epoch with the same power and rank:")
print(part2.groupby("table").agg(records=("match", "size"), matched=("match", "sum"), largest_diff=("max_diff", "max"),
                                 median_r=("r", "median"), slope_min=("slope", "min"), slope_max=("slope", "max")).to_string())
print("Unmatched Anxa1 traces:")
print(part2[(part2.line == "Anxa1") & ~part2.match].to_string(index=False))
m4398 = by_mouse[4398]
print(f"Mouse 4398: last onset at {m4398['start_time'][-1]:.2f} s, processed series ends at {m4398['t'][-1]:.2f} s")

# (3) Evoked integrals
code_auc = {s["mouse"]: lrrk2.evoked_integrals(s, "code") * lrrk2.FS for s in sessions}  # in samples, as the code
rows = []
for key in ("6E", "S7C", "S8E"):
    for d, s, j, idx in rank_match(src[key]):
        four = np.where(np.isclose(s["power_in_mW"], 4.0))[0][list(idx).index(j)]
        rows.append({"table": key, "line": s["line"], "mouse": d["mouse"], "power": d["power"], "published": d["value"],
                     "ours": code_auc[d["mouse"]][j], "ours_4mW_same_rank": code_auc[d["mouse"]][four]})
part3 = pd.DataFrame(rows)
factor = float(np.nanmedian(part3.published / part3.ours))
part3["match"] = np.abs(part3.published - factor * part3.ours) <= 1e-4 * np.abs(part3.published)
part3["match_if_4mW"] = np.abs(part3.published - factor * part3.ours_4mW_same_rank) <= 1e-4 * np.abs(part3.published)
print(f"\n(3) Evoked integrals: common factor published / ours (code's window, in samples) = {factor:.6g}")
print(part3.groupby(["table", "power"]).agg(records=("match", "size"), matched=("match", "sum"),
                                            matched_if_4mW=("match_if_4mW", "sum")).to_string())
print("Calb1 records: ratio published / ours from", part3[part3.line == "Calb1"].eval("published / ours").min().round(6),
      "to", part3[part3.line == "Calb1"].eval("published / ours").max().round(6))
print("Unmatched Anxa1 records:", part3[(part3.line == "Anxa1") & ~part3.match][["table", "mouse", "power", "published", "ours"]].to_dict("records"))

# (4) Spontaneous transients
measures = {ex: {s["mouse"]: lrrk2.spontaneous_transients(s, ex) for s in sessions} for ex in ("methods", "code_as_written")}
rows = []
for key in ("6F", "S8C"):
    pub = pd.DataFrame(src[key])
    for m, g in pub.groupby("mouse", sort=False):
        row = {"table": key, "line": line_of[key], "mouse": m, "published": len(g)}
        for ex in ("methods", "code_as_written"):
            auc, peaks, means = measures[ex][m]
            row[f"n_{ex}"] = len(auc)
            for name, v in (("integral", auc), ("mean", means)):
                ratio = g.value.values / v if len(v) == len(g) else np.array([np.nan])
                row[f"{name}_ratio_{ex}"] = np.median(ratio)
                row[f"{name}_ratio_cv_{ex}"] = np.std(ratio) / np.mean(ratio)
        rows.append(row)
part4 = pd.DataFrame(rows)
for ex in ("methods", "code_as_written"):
    for name in ("integral", "mean"):
        f4 = np.nanmedian(part4[f"{name}_ratio_{ex}"])
        ok = []
        for _, r in part4.iterrows():
            pub = pd.DataFrame(src[r.table]).query("mouse == @r.mouse").value.values
            v = measures[ex][r.mouse][0 if name == "integral" else 2]
            ok.append(len(pub) == len(v) and bool(np.allclose(np.sort(pub), f4 * np.sort(v), rtol=1e-2, atol=0)))
        part4[f"match_{name}_{ex}"] = ok
        print(f"(4) exclusion '{ex}', {name} of each transient: common factor {f4:.6g}; mice whose count and sorted values match",
              "within 1%:", int(np.sum(ok)), "of", len(ok), "(Anxa1", int(np.sum(np.array(ok)[part4.line == 'Anxa1'])), "of 15)")
cols = ["table", "mouse", "published", "n_methods", "n_code_as_written", "integral_ratio_cv_methods", "mean_ratio_methods",
        "mean_ratio_cv_methods"]
print(part4[cols].to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("Transients: published", part4.published.sum(), "; here, Methods exclusion", part4.n_methods.sum(),
      "; code's exclusion as written", part4.n_code_as_written.sum())
print("Count equal to the published count, Methods exclusion:", part4.groupby("line").apply(lambda g: f"{int((g.published == g.n_methods).sum())} of {len(g)}").to_dict())

score = {"mice": float(mouse_level.mean()), "traces": float(part2.match.mean()), "evoked integrals": float(part3.match.mean()),
         "spontaneous integrals (best window)": float(max(part4.match_integral_methods.mean(), part4.match_integral_code_as_written.mean()))}
print("\nFraction of records that match, per part, as the criterion scores them:", {k: round(v, 3) for k, v in score.items()})
print("Variant, part (4) with the mean dF/F of each transient:",
      round(float(max(part4.match_mean_methods.mean(), part4.match_mean_code_as_written.mean())), 3))

# (6) The paper's statistics recomputed from its own published values, to check the source data against the text.
pubdf = {k: pd.DataFrame(src[k]) for k in ("6E", "6F", "S7C", "S8C", "S8E")}
rows = []
for key, power, paper_p in (("6E", 0.5, "1.10e-20"), ("S7C", 4.0, "2.39e-16"), ("S8E", 0.5, "0.0002"), ("S8E", 4.0, "0.001")):
    g = pubdf[key][pubdf[key].power == power]
    gs, wt = g[g.group == "GS"].value, g[g.group == "WT"].value
    rows.append({"panel": key, "power": power, "n_GS": len(gs), "n_WT": len(wt), "p_one_tailed": lrrk2.one_tailed_less(gs, wt)[1],
                 "paper_p": paper_p, "cohens_d": lrrk2.cohens_d(wt, gs)})
for key, paper_p in (("6F", "2.75e-177"), ("S8C", "0.98")):
    g = pubdf[key]
    gs, wt = g[g.group == "GS"].value, g[g.group == "WT"].value
    rows.append({"panel": key, "power": np.nan, "n_GS": len(gs), "n_WT": len(wt), "p_one_tailed": lrrk2.one_tailed_less(gs, wt)[1],
                 "paper_p": paper_p, "cohens_d": lrrk2.cohens_d(wt, gs)})
print("\n(6) Statistics from the published values themselves (paper's Cohen's d at 0.5 mW: Anxa1 2.05, Calb1 0.76):")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

To see why the Calb1 records differ, the dF/F is recomputed from the raw 470 nm signal in the dandiset with the
code's binning and baseline correction, at the code's background fraction (0.85 of the sliding 8th percentile) and at
other fractions, and compared with the published triggered traces. The edges of the sliding percentile are not
reproduced exactly, so the comparison with the stored series leaves out the first and last 30 s.

In [ ]:
# (5) Why the Calb1 records differ: dF/F recomputed from the raw 470 nm signal in the dandiset with the code's
# processing, and with other background fractions, against the published triggered traces.
rows = []
for s in sessions:
    binned = lrrk2.raw_binned(s)
    key = "6C" if s["line"] == "Anxa1" else "S8D"
    pairs = [(d, j) for d, ss, j, idx in rank_match([d for d in src[key] if d["mouse"] == s["mouse"]])]
    stored_diff = np.abs(lrrk2.dff_from_raw(binned) - s["dff"])[3000:-3000].max()
    best = None
    for frac in np.round(np.arange(0.0, 0.951, 0.05), 2):
        dff = lrrk2.dff_from_raw(binned, frac)
        err = np.median([np.abs(dff[s["onset"][j] - 100:s["onset"][j] + 150] - d["trace"]).max() for d, j in pairs])
        if frac == 0.85:
            at_085 = err
        if best is None or err < best[1]:
            best = (frac, err)
    rows.append({"mouse": s["mouse"], "line": s["line"], "group": s["group"], "stored_vs_recomputed_max_diff": stored_diff,
                 "median_diff_at_0.85": at_085, "best_fraction": best[0], "median_diff_at_best": best[1]})
raw_check = pd.DataFrame(rows).sort_values(["line", "group", "mouse"])
print("Largest difference between the stored dF/F and dF/F recomputed from raw with the code's 0.85 (away from the edges),",
      "and the median over a mouse's published traces of the largest difference to dF/F recomputed with background fraction",
      "0.85 and with the best fraction of 0 to 0.95 (% dF/F):")
print(raw_check.round(3).to_string(index=False))

**Result: not reproduced.** All 27 mice in the source data are in the dandiset with the same genotype and line. For
Anxa1 the link holds: 120 of 120 triggered traces of Figure 6C and 479 of 480 of Supplementary Figure 7B equal the
dandiset's dF/F from 1 s before onset (largest difference 5.7e-13 in Figure 6C), the published evoked integrals equal
the code's measure times 0.0001 for all Anxa1 records except one (119 of 120 at 4 mW), and the transient counts match
for 15 of 15 mice. The two Anxa1 misses are the last 4 mW stimulation of mouse 4398, whose processed series ends 0.8 s
after onset. For Calb1 none of the 384 traces and none of the 384 integrals match, and the transient counts match for
1 of 12 mice. dF/F recomputed from the raw signal with the code's processing reproduces the stored Calb1 series (largest
difference 1.47 % dF/F) but not the published traces (median differences 66 to 519 % dF/F at the code's background
fraction, and 22 to 88 at the best fraction for each mouse). The published spontaneous values are 0.005 times the mean
dF/F over each transient (ratio 0.004994 to 0.004999 in the 15 Anxa1 mice), not its integral, so part (4) as the
criterion fixed it matches for no mouse. Scored as the criterion sets out, the four parts match for 1.0, 0.609, 0.609
and 0 of their records; with the transient mean in part (4), 0.481. The Figure 6E integrals match the 0.5 mW
epochs (120 of 120) and none of the 4 mW ones, so the figure uses the power its legend states. The published values
reproduce the paper's statistics (Figure 6E p = 1.1e-20, Supplementary Figure 8C p = 0.989, Cohen's d 2.06 and 0.765).

### Evoked integrals

Integrals for every train under the four windows: the Methods' light-on period (main), the code's -0.1 to 1.0 s
(`code`), the window `chunking_index.m` computes as written, starting 1 s after onset (`code_as_written`), and a fixed
0 to 0.5 s (`fixed`). All use the mean from 1.0 to 0.1 s before onset as baseline (shifted with the window for
`code_as_written`).

In [ ]:
def evoked_table(window):
    rows = []
    for s in sessions:
        values = lrrk2.evoked_integrals(s, window)
        for j, v in enumerate(values):
            rows.append({"mouse": s["mouse"], "line": s["line"], "group": s["group"], "power": s["power_in_mW"][j],
                         "train_s": round(s["stop_time"][j] - s["start_time"][j], 2), "value": v})
    return pd.DataFrame(rows)


windows = ["methods", "code", "code_as_written", "fixed"]
evoked = {w: evoked_table(w) for w in windows}
print("Evoked integrals (% dF/F x s) that could not be computed (window outside the recording):",
      {w: int(evoked[w].value.isna().sum()) for w in windows})


def group_test(df):
    gs, wt = df[df.group == "GS"].value.dropna(), df[df.group == "WT"].value.dropna()
    t, p = lrrk2.one_tailed_less(gs, wt)
    return {"n_GS": len(gs), "n_WT": len(wt), "mean_GS": gs.mean(), "mean_WT": wt.mean(),
            "t": t, "p_one_tailed": p, "cohens_d": lrrk2.cohens_d(wt, gs)}


def per_mouse_tests(df, alpha):
    wt = df[df.group == "WT"].value.dropna()
    rows = []
    for m, g in df[df.group == "GS"].groupby("mouse"):
        v = g.value.dropna()
        row = {"mouse": m, "n": len(v), "mean": v.mean(), "WT_mean": wt.mean()}
        for name, equal_var, tails in (("a_welch_1t", False, 1), ("b_welch_2t", False, 2),
                                       ("c_student_2t", True, 2), ("d_student_1t", True, 1)):
            p = (lrrk2.one_tailed_less(v, wt, equal_var)[1] if tails == 1 else lrrk2.two_tailed(v, wt, equal_var)[1])
            row[f"p_{name}"] = p
            row[f"sig_{name}"] = bool(p < alpha and v.mean() < wt.mean())
        rows.append(row)
    return pd.DataFrame(rows)

### fig6e-anxa1-evoked-reduced

> Optogenetically evoked dopamine release in the dorsolateral striatum (Anxa1+ projection region) at 0.5 mW is lower in
> LRRK2-G2019S than in LRRK2-WT mice (64 against 56 stimulations, p = 1.10e-20, one-tailed two-sample t-test).

The plot shows the mean dF/F (shading: SEM over stimulations) at 0.5 mW, with the 405 nm control dashed, and the
light-on period shaded. All points are within the axes; the ranges are printed.

In [ ]:
anxa = evoked["methods"].query("line == 'Anxa1' and power == 0.5")
main = group_test(anxa)
print("Anxa1, 0.5 mW, Methods window (light-on period):")
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in main.items()})
print(f"one-tailed p = {main['p_one_tailed']:.3g} (paper 1.10e-20)")
print("\nPer mouse (mean and range of the eight integrals, train length):")
print(anxa.groupby(["group", "mouse"]).agg(n=("value", "size"), mean=("value", "mean"), min=("value", "min"),
                                           max=("value", "max"), train_s=("train_s", "first")).round(2).to_string())
print("\nVariants (window, power): counts, means, one-tailed p, Cohen's d")
rows = []
for w in windows:
    for q in (0.5, 4.0):
        r = group_test(evoked[w].query("line == 'Anxa1' and power == @q"))
        rows.append({"window": w, "power": q, **r})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nVariant without the two G2019S mice whose 0.5 mW trains are 0.33 s long (3742, 4885), Methods window:")
r = group_test(anxa[~anxa.mouse.isin([3742, 4885])])
print({k: (f"{v:.3g}" if isinstance(v, float) else v) for k, v in r.items()})
print("Variant with one value per mouse (mouse means), Methods window: Welch one-tailed p =",
      f"{lrrk2.one_tailed_less(anxa[anxa.group == 'GS'].groupby('mouse').value.mean(), anxa[anxa.group == 'WT'].groupby('mouse').value.mean(), False)[1]:.3g}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
tt = np.arange(-100, 150) / lrrk2.FS
for ax, line in zip(axes, ("Anxa1", "Calb1")):
    for group, colour in (("GS", "tab:blue"), ("WT", "tab:gray")):
        tr, iso = [], []
        for s in sessions:
            if s["line"] != line or s["group"] != group:
                continue
            for j in np.where(s["power_in_mW"] == 0.5)[0]:
                tr.append(lrrk2.code_chunk(s, j))
                on = s["onset"][j]
                iso.append(s["dff405"][on - 100:on + 150])
        tr, iso = np.array(tr), np.array(iso)
        m, se = tr.mean(0), tr.std(0, ddof=1) / np.sqrt(len(tr))
        ax.plot(tt, m, color=colour, label=f"{group} (n = {len(tr)})")
        ax.fill_between(tt, m - se, m + se, color=colour, alpha=0.3)
        ax.plot(tt, iso.mean(0), color=colour, ls="--", lw=1)
        print(f"{line} {group}: mean trace from {m.min():.1f} to {m.max():.1f} % dF/F; 405 nm mean from {iso.mean(0).min():.1f} to {iso.mean(0).max():.1f}")
    ax.axvspan(0, 0.5, color="red", alpha=0.1)
    ax.set_title(f"{line}, 0.5 mW (dashed: 405 nm)")
    ax.set_xlabel("time from onset (s)")
    ax.set_ylabel("dF/F (%)")
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()

**Result: reproduced.** At 0.5 mW there are 64 GS and 56 WT stimulations, as in the paper. The mean integral over the
light-on period is 48.8 against 128.0 % dF/F x s, one-tailed p = 1.16e-22 (paper 1.10e-20), Cohen's d 2.21. With the
code's window p = 1.1e-20, the paper's value. The reduction holds at 4 mW (p = 9.9e-18), without the two GS mice given
0.33 s trains (p = 4.16e-17), and with one value per mouse (Welch p = 0.000434). The window `chunking_index.m` computes
as written reverses the sign (one-tailed p = 1, means -11.8 and -33), as it starts 1 s after onset. Every GS mouse
except 3743 has a lower mean than every WT mouse (GS 3.28 to 82.33 without 3743, WT 90.86 to 190.21).

### fig6e-anxa1-evoked-per-mouse

> Seven of eight individual G2019S mice show significantly reduced evoked dopamine release at 0.5 mW compared with the
> WT group; mouse 3743 does not.

In [ ]:
pm = per_mouse_tests(anxa, 0.05 / 8)
print("Each Anxa1 G2019S mouse against all WT integrals, 0.5 mW, Methods window; significant when p < 0.05/8 and the mean is lower.")
print(pm.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nSignificant mice per reading:", {c[4:]: int(pm[c].sum()) for c in pm.columns if c.startswith("sig_")})
print("Not significant under each reading:", {c[4:]: pm.loc[~pm[c], "mouse"].tolist() for c in pm.columns if c.startswith("sig_")})
print("Paper's p-values (legend): 4885 2.57e-10, 4651 0.0004, 4652 7.36e-05, 3742 4.14e-15, 3743 0.478, 4004 2.81e-13, 4005 1.72e-14, 4007 2.36e-12")
print("\nVariants (other windows, and 4 mW): significant mice per reading")
for w in windows:
    for q in (0.5, 4.0):
        v = per_mouse_tests(evoked[w].query("line == 'Anxa1' and power == @q"), 0.05 / 8)
        print(f"  {w:16s} {q} mW:", {c[4:]: int(v[c].sum()) for c in v.columns if c.startswith("sig_")},
              "not significant (a):", v.loc[~v.sig_a_welch_1t, "mouse"].tolist())

**Result: reproduced.** Under all four readings 7 of 8 GS mice are below the WT group at p < 0.05/8, and 3743 is the
exception (mean 125 against 128, Welch one-tailed p = 0.362), as in the paper. The Student one-tailed p-values are of
the same order as the legend's, for example 1.01e-04 against 4.0e-04 for 4651 and 3.73e-16 against 4.14e-15 for 3742.
With the code's window the count is 7 under every reading, and at 4 mW it is 7 under the Welch readings.

### supp7c-anxa1-evoked-all-powers

> The reduction of evoked dopamine release in Anxa1 G2019S mice holds at each of the four stimulation powers, 0.1, 0.5,
> 1 and 4 mW (p from 2.39e-16 to 1.10e-20).

In [ ]:
rows = []
for w in windows:
    for q in lrrk2.PAPER_POWERS:
        rows.append({"window": w, "power": q, **group_test(evoked[w].query("line == 'Anxa1' and power == @q"))})
s7 = pd.DataFrame(rows)
print("Anxa1, evoked integrals per power; paper p: 0.1 mW 1.29e-15, 0.5 mW 1.10e-20, 1 mW 1.22e-17, 4 mW 2.39e-16")
print(s7.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
main7 = s7[s7.window == "methods"]
print("\nMethods window: powers with G2019S lower at one-tailed p < 0.05:",
      int(((main7.p_one_tailed < 0.05) & (main7.mean_GS < main7.mean_WT)).sum()), "of 4")

**Result: reproduced.** GS integrals are lower at all four powers, one-tailed p from 3.13e-15 (0.1 mW) to 1.16e-22
(0.5 mW) with the Methods' window. With the code's window the p-values at 0.1, 0.5 and 1 mW are 1.29e-15, 1.1e-20 and
1.22e-17, the paper's values; at 4 mW it is 5.62e-16 against 2.39e-16, as the dandiset has 55 WT stimulations there in
place of 56.

### Spontaneous transients

Transients for every mouse, with the Methods' exclusion of 1 s before to 1.5 s after each onset (main) and with the
window the code removes as written (1 s to 3.5 s after onset). Each transient's integral, peak and mean dF/F are kept.

In [ ]:
def spont_table(exclusion):
    rows = []
    for s in sessions:
        auc, peaks, means = lrrk2.spontaneous_transients(s, exclusion)
        rows += [{"mouse": s["mouse"], "line": s["line"], "group": s["group"], "value": a, "peak": p, "mean": m}
                 for a, p, m in zip(auc, peaks, means)]
    return pd.DataFrame(rows)


spont = {ex: spont_table(ex) for ex in ("methods", "code_as_written")}
for ex, df in spont.items():
    print(f"{ex}: {len(df)} transients over {df.mouse.nunique()} mice; integral (% dF/F x s) from {df.value.min():.2f} to {df.value.max():.1f}")

### fig6f-anxa1-spontaneous-reduced

> Spontaneous dopamine transients in the dorsolateral striatum are smaller in G2019S than in WT mice (1064 against 798
> transients, p = 2.75e-177, one-tailed two-sample t-test), with 57 to 285 transients per mouse.

In [ ]:
sa = spont["methods"].query("line == 'Anxa1'")
main_s = group_test(sa)
print("Anxa1 spontaneous transients, Methods exclusion:")
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in main_s.items()})
print(f"one-tailed p = {main_s['p_one_tailed']:.3g} (paper 2.75e-177; counts 1064 G2019S, 798 WT)")
per = sa.groupby(["group", "mouse"]).agg(n=("value", "size"), mean=("value", "mean"), median=("value", "median"),
                                          max=("value", "max"), mean_peak=("peak", "mean"))
print("\nPer mouse:")
print(per.round(2).to_string())
print("Transients per mouse from", per.n.min(), "to", per.n.max(), "(paper 57 to 285; criterion 51.3 to 313.5)")
v = spont["code_as_written"].query("line == 'Anxa1'")
r = group_test(v)
print("\nVariant, the code's exclusion window as written:", {k: (f"{x:.3g}" if isinstance(x, float) else x) for k, x in r.items()})
print("Variant, one value per mouse (mouse means), Welch one-tailed p =",
      f"{lrrk2.one_tailed_less(per.loc['GS', 'mean'], per.loc['WT', 'mean'], False)[1]:.3g}")
print("Variant, transient peaks in place of integrals: one-tailed p =",
      f"{lrrk2.one_tailed_less(sa[sa.group == 'GS'].peak, sa[sa.group == 'WT'].peak)[1]:.3g}")
r = group_test(sa.assign(value=sa["mean"]))
print("Variant, the mean dF/F over each transient (the measure the source data hold, see source-data-link):",
      {k: (f"{x:.3g}" if isinstance(x, float) else x) for k, x in r.items()})

**Result: reproduced.** The Methods' detection gives 1064 GS and 798 WT transients, the paper's counts, with 57 to 285
per mouse, the paper's range. Mean integrals are 131.9 against 302.9 % dF/F x s, one-tailed p = 4.85e-72 (paper
2.75e-177). The published values are the mean dF/F over each transient (see `source-data-link`); with that measure
p = 2.23e-177. With one value per mouse the difference gives Welch p = 0.0318, and the code's exclusion window as
written gives p = 1.3e-68 on 648 and 363 transients.

### fig6f-anxa1-spontaneous-per-mouse

> Six of eight individual G2019S mice show significantly smaller spontaneous dopamine transients than the WT group;
> mice 4651 and 3743 do not.

In [ ]:
pms = per_mouse_tests(sa, 0.05 / 8)
print("Each Anxa1 G2019S mouse's transient integrals against all WT transients, Methods exclusion; threshold 0.05/8.")
print(pms.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nSignificant mice per reading:", {c[4:]: int(pms[c].sum()) for c in pms.columns if c.startswith("sig_")})
print("Not significant under each reading:", {c[4:]: pms.loc[~pms[c], "mouse"].tolist() for c in pms.columns if c.startswith("sig_")})
print("Paper's p-values (legend): 4885 1.44e-49, 4651 0.008, 4652 6.21e-13, 3742 5.28e-88, 3743 0.97, 4004 1.24e-95, 4005 4.13e-109, 4007 5.00e-107")
v = per_mouse_tests(sa.assign(value=sa["mean"]), 0.05 / 8)
print("\nVariant, the mean dF/F over each transient:", {c[4:]: int(v[c].sum()) for c in v.columns if c.startswith("sig_")},
      "not significant (a):", v.loc[~v.sig_a_welch_1t, "mouse"].tolist())
print(v[["mouse", "mean", "WT_mean", "p_a_welch_1t", "p_d_student_1t"]].to_string(index=False, float_format=lambda x: f"{x:.3g}"))
v = per_mouse_tests(spont["code_as_written"].query("line == 'Anxa1'"), 0.05 / 8)
print("\nVariant, code's exclusion as written:", {c[4:]: int(v[c].sum()) for c in v.columns if c.startswith("sig_")},
      "not significant (a):", v.loc[~v.sig_a_welch_1t, "mouse"].tolist())

**Result: partially reproduced.** By the integral, 5 of 8 GS mice are below the WT group under every reading. The
exceptions are 3743 (Welch one-tailed p = 0.431), 4651, whose mean (417) is above the WT mean (303), and 4652
(p = 0.0128). With the mean dF/F over each transient, the measure of the published values, reading (d) gives 6 of 8
with 3743 and 4651 as the exceptions, and p-values close to the legend's (4651 0.00819 against 0.008, 3743 0.965
against 0.97, 3742 5.38e-88 against 5.28e-88). The Welch readings give 7 there, as 4651 reaches p = 1.5e-05, so the
paper's count follows the one-tailed Student test of its legend, not the Welch test of its Methods.

### supp8c-calb1-spontaneous-not-reduced

> In the dorsomedial striatum (Calb1+ projection region), spontaneous dopamine transients are not reduced in G2019S
> mice (970 against 682 transients, p = 0.98).

In [ ]:
sc = spont["methods"].query("line == 'Calb1'")
r = group_test(sc)
print("Calb1 spontaneous transients, Methods exclusion (paper: 970 G2019S, 682 WT, one-tailed p = 0.98):")
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()})
print(f"one-tailed p (G2019S lower) = {r['p_one_tailed']:.3g}; difference of means (WT - G2019S) = {r['mean_WT'] - r['mean_GS']:.3f} % dF/F x s")
print("Two-tailed p =", f"{lrrk2.two_tailed(sc[sc.group == 'GS'].value, sc[sc.group == 'WT'].value)[1]:.3g}")
per = sc.groupby(["group", "mouse"]).agg(n=("value", "size"), mean=("value", "mean"), median=("value", "median"))
print("\nPer mouse:")
print(per.round(3).to_string())
v = spont["code_as_written"].query("line == 'Calb1'")
r2 = group_test(v)
print("\nVariant, the code's exclusion window as written:", {k: (f"{x:.3g}" if isinstance(x, float) else x) for k, x in r2.items()})
print("Variant, mouse means, Welch one-tailed p =",
      f"{lrrk2.one_tailed_less(per.loc['GS', 'mean'], per.loc['WT', 'mean'], False)[1]:.3g}")
r3 = group_test(sc.assign(value=sc["mean"]))
print("Variant, the mean dF/F over each transient:", {k: (f"{x:.3g}" if isinstance(x, float) else x) for k, x in r3.items()})

**Result: reproduced.** By the paper's test, GS transients are not significantly smaller: one-tailed p = 0.0509 on 896
GS and 635 WT transients (both within tolerance of 970 and 682), difference of means 21.1 % dF/F x s, Cohen's d 0.085.
The result sits at the threshold. With the mean dF/F over each transient, the measure of the published values, GS are
lower at p = 2.25e-09 (d = 0.306), and with the code's exclusion window as written at p = 1.81e-05; with one value per
mouse, Welch p = 0.402. The dandiset's Calb1 dF/F is not the trace the paper analysed (see `source-data-link`), and
the published values give p = 0.989.

### supp8e-calb1-evoked-reduced

> Evoked dopamine release in the dorsomedial striatum from Calb1+ neuron stimulation is reduced in G2019S mice at each
> of four powers (56 against 40 stimulations per power, p from 6.45e-05 to 0.001).

In [ ]:
rows = []
for w in windows:
    for q in lrrk2.PAPER_POWERS:
        rows.append({"window": w, "power": q, **group_test(evoked[w].query("line == 'Calb1' and power == @q"))})
s8 = pd.DataFrame(rows)
print("Calb1, evoked integrals per power; paper p: 0.1 mW 6.45e-05, 0.5 mW 0.0002, 1 mW 0.0001, 4 mW 0.001 (56 against 40)")
print(s8.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
main8 = s8[s8.window == "methods"]
print("\nMethods window: powers with G2019S lower at one-tailed p < 0.05:",
      int(((main8.p_one_tailed < 0.05) & (main8.mean_GS < main8.mean_WT)).sum()), "of 4")
cal = evoked["methods"].query("line == 'Calb1' and power == 0.5")
print("\nPer mouse, 0.5 mW, Methods window:")
print(cal.groupby(["group", "mouse"]).value.agg(["size", "mean", "min", "max"]).round(2).to_string())

**Result: reproduced.** There are 56 GS and 40 WT stimulations per power, as in the paper. GS are lower at all four
powers, one-tailed p from 4.38e-07 (4 mW) to 6.29e-10 (0.5 mW), smaller p-values than the paper's 6.45e-05 to 0.001;
the published integrals do not come from the dandiset's Calb1 dF/F (see `source-data-link`).

### supp8e-calb1-few-mice-reduced

> Only 2 of 7 G2019S mice show reduced Calb1+ evoked dopamine release relative to the WT group.

In [ ]:
pmc = per_mouse_tests(cal, 0.05 / 7)
print("Each Calb1 G2019S mouse against all WT integrals, 0.5 mW, Methods window; threshold 0.05/7.")
print(pmc.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nSignificant mice per reading:", {c[4:]: int(pmc[c].sum()) for c in pmc.columns if c.startswith("sig_")})
print("Significant under each reading:", {c[4:]: pmc.loc[pmc[c], "mouse"].tolist() for c in pmc.columns if c.startswith("sig_")})
print("\nVariants (other powers and windows): significant mice per reading")
for w in windows:
    for q in lrrk2.PAPER_POWERS:
        v = per_mouse_tests(evoked[w].query("line == 'Calb1' and power == @q"), 0.05 / 7)
        print(f"  {w:16s} {q} mW:", {c[4:]: int(v[c].sum()) for c in v.columns if c.startswith("sig_")})

**Result: partially reproduced.** At 0.5 mW, the number of GS mice below the WT group at p < 0.05/7 is 7 (Welch
one-tailed), 6 (Welch two-tailed) and 4 (Student, either tail), against the paper's 2; the closest reading gives 4.
Student tests give 2 at 0.1 and 1 mW and 1 or 3 at 4 mW, so the count depends on power and test. Mouse 740 (mean 17.3
against 131) is below the WT group under every reading.

### cohens-d-anxa1-larger

> The G2019S reduction of evoked release at 0.5 mW is much larger for Anxa1+ (Cohen's d = 2.05) than for Calb1+
> stimulation (d = 0.76).

In [ ]:
rows = []
for w in windows:
    for q in lrrk2.PAPER_POWERS:
        da = lrrk2.cohens_d(*[evoked[w].query("line == 'Anxa1' and power == @q and group == @g").value.dropna() for g in ("WT", "GS")])
        dc = lrrk2.cohens_d(*[evoked[w].query("line == 'Calb1' and power == @q and group == @g").value.dropna() for g in ("WT", "GS")])
        rows.append({"window": w, "power": q, "d_Anxa1": da, "d_Calb1": dc, "ratio": da / dc})
cd = pd.DataFrame(rows)
print("Cohen's d (WT minus G2019S over the pooled SD); paper at 0.5 mW: Anxa1 2.05, Calb1 0.76")
print(cd.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
m = cd.query("window == 'methods' and power == 0.5").iloc[0]
print(f"\nMethods window, 0.5 mW: Anxa1 d = {m.d_Anxa1:.3f} (tolerance 1.845 to 2.255), Calb1 d = {m.d_Calb1:.3f} (0.684 to 0.836), ratio {m.ratio:.2f}")

**Result: not reproduced.** At 0.5 mW, Cohen's d is 2.21 for Anxa1, within tolerance of 2.05, and 1.40 for Calb1,
against 0.76 (tolerance 0.684 to 0.836); their ratio is 1.58, below the factor of 2 the criterion asks. With the code's
window d is 2.06 and 1.25. The published integrals give 2.06 and 0.765 (see `source-data-link`), so the paper's values
follow from its own data, and the Calb1 difference is consistent with the Calb1 dF/F in the dandiset not being the one
the paper analysed. The Anxa1 effect is larger than the Calb1 effect at every power under the Methods, code and fixed
windows (ratio 1.29 to 1.74).

### Blocked claims

These claims rest on data that are not in the dandiset, which holds only the photometry of Figure 6 and Supplementary
Figures 7 and 8. The data exist elsewhere: images in the BioImage Archive (S-BIAD2070, S-BIAD2071, S-BIAD2072,
S-BIAD2074), mass spectrometry in PRIDE (PXD065003, PXD065006, PXD065125), and per-section and per-blot values in the
source data file on Zenodo. The registry judges the dandiset, so they are recorded as blocked (`data_missing`).

| Claim | Paper | Data |
|---|---|---|
| `fig1d-lrrk2-enriched-vulnerable` | LRRK2 higher in Aldh1a1+ SNc neurons | immunostaining images |
| `fig2b-rab3-phosphorylation` | 289 phosphopeptides change with MLi-2 | phosphoproteomics |
| `fig2eg-prab3-increased` | pRAB3A and pRAB3C up in G2019S | PhosTag western blots |
| `fig3d-rim-binds-unphosphorylated` | RIM1 and RIM2 prefer unphosphorylated RAB3A | affinity purification MS |
| `fig4d-fewer-active-zones` | Fewer bassoon+ TH+ synaptosomes | confocal images |
| `fig4i-rim1-reduced` | RIM1 reduced in synaptosomes | western blots |
| `fig5-bassoon-vulnerable-axons` | Bassoon clusters change in vulnerable axons | SIM images |
| `fig5j-no-axon-loss` | No loss of TH+ fibres | confocal images |

## Summary

| Claim | Outcome |
|---|---|
| `source-data-link` | not reproduced (Anxa1 links; Calb1 does not) |
| `fig6e-anxa1-evoked-reduced` | reproduced |
| `fig6e-anxa1-evoked-per-mouse` | reproduced |
| `supp7c-anxa1-evoked-all-powers` | reproduced |
| `fig6f-anxa1-spontaneous-reduced` | reproduced |
| `fig6f-anxa1-spontaneous-per-mouse` | partially reproduced |
| `supp8c-calb1-spontaneous-not-reduced` | reproduced (p = 0.0509, at the threshold) |
| `supp8e-calb1-evoked-reduced` | reproduced |
| `supp8e-calb1-few-mice-reduced` | partially reproduced |
| `cohens-d-anxa1-larger` | not reproduced |
| Eight claims of Figures 1 to 5 | blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences
from the paper's method.

How the analysis differs from the paper's:

- The evoked integral is taken over the light-on period, as the Methods state; the paper's numbers come from the code's
  window, 0.1 s before to 1.0 s after onset (its published integrals equal that measure times 0.0001). Both are shown.
- Spontaneous transients are measured by their integral, as the Methods state; the paper's published values are 0.005
  times the mean dF/F over each transient. Both are shown.
- Per-mouse tests are shown under four readings, as the Methods (Welch), the legends (one-tailed) and the code
  (Student, two-tailed) disagree.
- The window `chunking_index.m` computes as written starts 1 s after onset; the source data show the paper used the
  intended window starting 1 s before onset. The literal window is shown as a variant only.

What was found about the dandiset:

- The Anxa1 recordings are those behind Figure 6 and Supplementary Figure 7: the published traces equal the dandiset's
  dF/F to within 1e-12, and the evoked integrals and transient counts match, except one stimulation of mouse 4398 that
  the processed series cuts off 0.8 s after onset.
- The Calb1 dF/F in the dandiset is not the trace behind Supplementary Figure 8. None of the 384 published traces or
  integrals match it, and recomputing from the raw signal with any single background fraction does not close the gap.
  The raw signal is there, but the processing the paper used for Calb1 is not, so the Calb1 results here are of the
  dandiset's processing, not the paper's.
- The epochs table describes one kind of train and four powers; the files hold two kinds of train and eight powers, and
  two Anxa1 GS mice received the paper's powers as 0.33 s trains of 9 ms pulses.
- The dF/F unit is given as `a.u.` for a series in percent; two trains have a pulse length of 0; the four excluded mice
  are left out without a note; and the paper does not cite the dandiset.

In [ ]:
print(f"total run time {time.time() - started:.0f} s")